# Model proposal for continous integration

Here we  show the proposal along with the final outupt:

- smooothing using the LOESS filter with a moving window of 7 deltas (L2)

<a id="table"></a>

## Table of Contents
- [Model explanation](#model-explanation)

    - [Outlier detecion](#outlier-detecion)
    - [NDVI Estimation](#ndvi-estimation)
    - [Linear delta interpolation](#linear-delta-interpolation)
    - [Smoothing](#Smoothing)
    - [Data to save](#what-needs-to-be-saved-for-the-model)
    - [Writing data](#what-the-model-will-write)

- [Case tested](#case-tested)

    - [Case 1: lowland broadleaf](#lowland-broadleaf)
    - [Case 2: highland broadleaf](#highland-broadleaf)
    - [Case 3: lowland evergreen](#lowland-evergreen)
    - [Case 4: highland broadleaf](#highland-evergreen)
    - [Case 5: fire-affected area](#fire)
    - [Case 6: nearby fire-affected area](#non-fire)
    - [Case 7: 2018 drought-affected area](#drought)
    - [Case 8: Vaia storm-affected area](#storm)

- [Area visualization](#areas)



<a id="model-explanation"></a>

[Go back to the table](#table)

## Model explanation

The model follows this path

Outlier detection -> estimation NDVI -> L1 linear interpolation -> L2 smoothing

<a id="outlier-detecion"></a>

[Go back to the table](#table)

### Outlier detecion

The outlier detecion will follows this rule

- check if the new observation is inside the bands
- if not, check if **one** extreme condition is met, if so it will be flagged as outlier. The extreme threshold depends on the distribution of the full time serie (we use the quantiles).
- check if the delta delta **and** the distance from the center of the bands are above the threshold, if so it will be flagged as outlier

if there is a pending potential outlier and the incoming data is a true value or a potential outler

- calculate the new delta delta, if it is above the threshold the potential outlier is flagged as outlier and the data is removed

<a id="ndvi-estimation"></a>

[Go back to the table](#table)

### NDVI Estimation

The estimation will be done when no observation are avaible or when the data is flagged as outlier.

The estimation is done based on the last delta multiplied by the exponential decay

<a id="linear-delta-interpolation"></a>

[Go back to the table](#table)

### Linear delta interpolation

The linear deltas interpolation between the lsat and second last observation and it will overwrite the NDVI estimation.

In case of a potential outlier is flagged as true value, we perform the linear interpolation between the second-most recent known observation and the potential outlier and from the potential outlier to the last known observation

<a id="Smoothing"></a>

[Go back to the table](#table)

### Smoothing 

To work at best, the data to smooth (so the deltas) has to be in between the smoothing window.

By choosing a window length of 7, we will smooth the fourth value.

The smoothing and L2 writing will be done iteratively between the last fifth and fourth observation in the smoothing window.

It will overwrite the L1 interpolation between the last thrid and fourth observation.

In case of a potential outlier is flagged as true value, we perform the smoothing, insert the delta in the proper position (as second-most recent observation), crop the window smoothing and re-perform the smoothing accordingly.

Proposal: in case of high variability in the data to smooth (as of evegreen areas), a rolling mean window with size 3 is performed on smoothed values.

In case of extreme negative events (NDVI less than 0.2 compared to interquantile range). The smoothing is performed on the NDVI values and not on the deltas. This represents the fact that vegetation may not follow the expected values given the extreme conditions.

<a id="what-needs-to-be-saved-for-the-model"></a>

[Go back to the table](#table)

## What needs to be saved for the model 

To work, the model requires 8 dates saved in 1 array

- The first 7 dates will be used to perform the smoothing, with the last of teh 7 dates will be the last date with an observed NDVI value
- The date 8th date will be used to store the potential outlier date

In case of no potential outlier, the 8th date will be assgigned the value "1900-01-01"

All this data has to be saved for each pixel

<a id="what-the-model-will-write"></a>

[Go back to the table](#table)

## What the model will write

The model will follow this logic for the data ingestion

- the data is na or is flagges as outlier
    - estimate using last delta and exponential decay = **write one value**

- the data is flagged as potential outlier
    - **write the potential outlier value**

- the data is flagged as true value
    - if a potential outlier is pending, check the value
        - if is confirmed as true value, linearly gapfill twice as explained above = **write between previous last known observation and last known observation**
        - perfrom smoothing twice as described above = **write between previous last fifth known observation and last fourth known observation**
    - if not 
        - perform a simple linear interpolation = **write between previous last known observation and last known observation**
        - perfrom a simple smoothing smoothing= **write between previous last fifth known observation and last fourth known observation**

### Potential backtrack and excess overwrite

With this approach, there is no backtrack and each observation is written at maximum 3 times (one per product) and each product is never overwritten twice

Here's a image summarising the model decision

<figure>
  <img src="fig/flowchart.png" alt="My Diagram">
</figure>

<a id="case-tested"></a>

[Go back to the table](#table)

## Case tested

We test this model on different biomes and known cases. Each case is represented by 25 pixels. All pixels are collected and arranged according to the following table. 

For each coordinate listed below, we select a square of 60 meters centered aorund the coordinates and we select 25 pixel from it

| Biome                                 | Coordinates (x, y)        | Pixel Range |
|---------------------------------------|---------------------------|-------------|
| Lowland broadleaf                     | 2694491.82, 1126023.20    | 0–24        |
| Highland broadleaf                    | 2692020.28, 1121443.47    | 25–49       |
| Lowland evergreen                     | 2761097.61, 1194613.45    | 50–74       |
| Highland evergreen                    | 2781537.00, 1182975.00    | 75–99       |
| Biscth fire affected area             | 2644029.37, 1134128.20    | 100–124     |
| Biscth fire nearby non-affected area  | 2644328.07, 1134342.81    | 125–149     |
| Drought-affected area                 | 2690025.48, 1287413.03    | 150–174     |
| Vaia storm affected area              | 2689564.74, 1154411.88    | 175–199     |

For each case, we will analyse 
- a pixel in detail, covireing all three products and the latency (difference between current date and smoothing)
- the smoothign of 24 pixels
- the area comparison between raw data and smoothing


In [10]:
# This is just to embed the web page and the videos
from IPython.display import IFrame, Video

<a id="lowland-broadleaf"></a>

[Go back to the table](#table)

## Case 1: lowland broadleaf

The broadleaf biomes selected are nicely represent and do not present any complication.

In [11]:
# area location
x, y = 2694491.82, 1126023.20

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

The lowland broadleaf area appears to be correctly smoothed.

There are no significant problem in the outlier detection and smoothing.

On winter 2024, we identified a reduction of NDVI across all points which is correctly captured.

<figure>
  <img src="../figure/broad_low_1.png" alt="My Diagram">
</figure>


<figure>
  <img src="../figure/broad_low_2.png" alt="My Diagram">
</figure>

<a id="highland-broadleaf"></a>

[Go back to the table](#table)

## Case 2: Highland broadleaf

In [12]:
# area location
x, y = 2692020.28, 1121443.47

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

The highland broadlead biomes appears to have almost no problem in detecting early summer NDVI values.

The lower and upper bands appear different between neighbouring pixels, especially during winter.

We are able to correctly identify and smooth the NDVI timeserie despite the scattering of data.

<figure>
  <img src="../figure/broad_high_1.png" alt="My Diagram">
</figure>


<figure>
  <img src="../figure/broad_high_2.png" alt="My Diagram">
</figure>

<a id="lowland-evergreen"></a>

[Go back to the table](#table)

## Case 3: Lowland evergreen

The lowland evergreen biome selected has high variability (as expcted). With the new set of parameters we are able to represent correctly.

In [13]:
# area location
x, y = 2761097.61, 1194613.45

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

The lowland evergreen selected area is composed by sparce vegetation and the canopy does not fully cover the ground. This may impact the NDVI values increasing the scattering of them.

There is no much variation between summer and winter NDVI values, which is expected considering that the area selected is a evergreen forset.

There is a sharp drop on winter on the upper and lower bands. This is caused by the fact that the first and last day of the double sigmoid function do not necessarly overlap. 

<figure>
  <img src="../figure/ever_low_1.png" alt="My Diagram">
</figure>


<figure>
  <img src="../figure/ever_low_2.png" alt="My Diagram">
</figure>

<a id="highland-evergreen"></a>

[Go back to the table](#table)

## Case 4: Highland evergreen

With the new set of parameters the interquantile range is very large during winter. For that reason we observe a sharp drop during that season. However, LOESS smoothing method has the ability to drastically smooth an extreme value (that we would flag if it was outside the IQR).

In [14]:
# area location
x, y = 2781537.00, 1182975.00 

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

In the evergreen biomes there is some scattering but we are able to follow the NDVI timeserie correctly

<figure>
  <img src="../figure/ever_high_1.png" alt="My Diagram">
</figure>


<figure>
  <img src="../figure/ever_high_2.png" alt="My Diagram">
</figure>

<a id="fire"></a>

[Go back to the table](#table)

## Case 5: Fire-affected area

With the new set of parameters we are able to correctly flag the drastic reduction in NDVI caused by the fire event. After the event we use the absoulte NDVI to smooth the values instead of the deltas because it is not garantee that the vegetation follows the expected behavior after this drastic event.

In [15]:
# area location
x, y = 2644029.37, 1134128.20 

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

<figure>
  <img src="../figure/fire_1.png" alt="My Diagram">
</figure>


<figure>
  <img src="../figure/fire_2.png" alt="My Diagram">
</figure>

<a id="non-fire"></a>

[Go back to the table](#table)

## Case 6: Nearby fire-affected area

The nreaby non affected area by the Bistch fire is clearly different and we (rightfully) do not see the fire event.

In [16]:
# area location
x, y = 2644328.07, 1134342.81

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)


Due to the sparce vege


<figure>
  <img src="../figure/non_fire_1.png" alt="My Diagram">
</figure>


<figure>
  <img src="../figure/non_fire_2.png" alt="My Diagram">
</figure>

<a id="drought"></a>

[Go back to the table](#table)

## Case 7: Drought affected area

We selected an area north of Schaffausen as illustrated in Fig. 3 in https://onlinelibrary.wiley.com/doi/10.1111/gcb.15360

This drought event mildy affected the vegetation but is possible to see a sharper drop at the summer 2018 season. We are able to correctly indentified the drought and non drought pixels.

In [17]:
# area location
x, y = 2690025.48, 1287413.03

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

<figure>
  <img src="../figure/drought_1.png" alt="My Diagram">
</figure>


<figure>
  <img src="../figure/drought_2.png" alt="My Diagram">
</figure>

<a id="storm"></a>

[Go back to the table](#table)

## Case 8: Storm Vaia affected area

The effect of Vaia storm is clearly visible and the NDVI is correctly flagged. It is super interesting to see the different NDVI series after the Vaia storm. The model correctly not flagged any of this different response despite using a nearly identical set of parameters.

The sparce vegetation here again influence the NDVI scattering.

In [18]:
# area location
x, y = 2689564.74, 1154411.88

# Construct the URL with your coordinates as center
url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# Display map in notebook
IFrame(url, width=1000, height=600)

<figure>
  <img src="../figure/storm_1.png" alt="My Diagram">
</figure>


<figure>
  <img src="../figure/storm_2.png" alt="My Diagram">
</figure>